<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

## Build the Feature Vector

Here I build my feature vector from the starter dataset. Each feature is engineered to be:

- **Knowable at decision time** — no future data leaks in
- **Clean** — missing values handled appropriately
- **Interpretable** — I can explain what each feature means

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Setup for Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== Building Feature Vector ===\n")
print(f"Total rows: {len(df):,}\n")

# === Define features (knowable at decision time) ===

# Numeric features (directly available)
numeric_features = [
    'avg_position',           # Current position
    'ctr',                    # Click-through rate
    'engagement_rate',        # Visitor engagement
    'content_age_days',       # Age of content
    'word_count',             # Content length
    'search_volume',          # Keyword demand
    'competition',            # Competition level
    'days_since_last_update'  # Freshness
]

# Categorical features (will be one-hot encoded)
categorical_features = [
    'content_type',           # Type of page
    'main_intent'             # User intent
]

# === Build feature matrix ===

# Start with numeric features
X_numeric = df[numeric_features].copy()

# Handle missing values:
# - avg_position: 0 means "no data" → replace with -1 to distinguish
X_numeric['avg_position'] = X_numeric['avg_position'].replace(0, -1)

# - For other numeric columns, fill with median
for col in numeric_features:
    if col != 'avg_position':
        median_val = X_numeric[col].median()
        X_numeric[col] = X_numeric[col].fillna(median_val)

# Handle categorical features (one-hot encode)
X_categorical = pd.get_dummies(df[categorical_features], drop_first=True)

# Combine
X = pd.concat([X_numeric, X_categorical], axis=1)

# === Create label ===
y = (df['trend_direction'] == 'down').astype(int)

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"Rows: {X.shape[0]}")
print(f"\nLabel distribution:")
print(f"  Declining (1): {y.sum():,} ({y.mean()*100:.1f}%)")
print(f"  Not declining (0): {(1-y).sum():,} ({(1-y.mean())*100:.1f}%)")

# Show first few rows
print("\n=== Sample Feature Vector (first 5 rows) ===")
X.head(5)

Working dir: /content/flyrank-ml-internship-starter
=== Building Feature Vector ===

Total rows: 30,000

Feature matrix shape: (30000, 13)
Features: 13
Rows: 30000

Label distribution:
  Declining (1): 16,262 (54.2%)
  Not declining (0): 13,738 (45.8%)

=== Sample Feature Vector (first 5 rows) ===


,avg_position,ctr,engagement_rate,content_age_days,word_count,search_volume,competition,days_since_last_update,content_type_feedly article,content_type_keyword article,main_intent_informational,main_intent_navigational,main_intent_transactional
0,10.6,0.76,5.88,187,3221.0,10.0,0.67,20,False,True,False,False,True
1,20.3,0.05,0.00,445,2481.0,90.0,0.01,25,False,True,True,False,False
2,36.5,0.09,0.00,141,3515.0,0.0,0.00,20,False,True,True,False,False
3,6.2,0.49,1.28,463,2877.0,10.0,0.00,22,False,True,False,False,False
4,44.0,0.13,0.00,263,2803.0,0.0,0.00,14,False,True,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature Notes

For each feature, I document:
- **What it means** — plain English
- **Missing handling** — how I dealt with nulls
- **Available when?** — is it knowable before prediction?

### Numeric Features

| Feature | Meaning | Missing Handling | Available When? |
|---|---|---|---|
| `avg_position` | Average search rank (lower = better) | 0 = "no data" → replaced with -1 to distinguish | Feature window (past) |
| `ctr` | Click-through rate | Filled with median | Feature window (past) |
| `engagement_rate` | Visitor engagement | Filled with median | Feature window (past) |
| `content_age_days` | Age of content in days | Filled with median | At decision point |
| `word_count` | Number of words on page | Filled with median | At decision point |
| `search_volume` | Monthly keyword searches | Filled with median | Feature window (past) |
| `competition` | How hard to rank (0-1) | Filled with median | Feature window (past) |
| `days_since_last_update` | Days since last update | Filled with median | At decision point |

### Categorical Features

| Feature | Meaning | Missing Handling | Available When? |
|---|---|---|---|
| `content_type` | Type of page (article, etc.) | One-hot encoded | At decision point |
| `main_intent` | User intent (informational, etc.) | One-hot encoded | At decision point |

### Leakage Prevention

🔴 **EXCLUDED FROM FEATURES:**
- `trend_direction` and `trend_pct` — **These are derived from the label!** Using them would leak the answer.

🟢 **FEATURES ARE SAFE BECAUSE:**
- All features come from the **feature window (past)**, NOT the target window (future)
- Content metadata (word_count, age) is fixed — knowable at decision time
- Performance metrics (position, CTR, engagement) are from the feature window

In [2]:
print("=== Feature Notes ===\n")

print("Numeric Features:")
for col in numeric_features:
    non_null = X_numeric[col].notna().sum()
    unique = X_numeric[col].nunique()
    min_val = X_numeric[col].min()
    max_val = X_numeric[col].max()
    print(f"  {col}:")
    print(f"    - Non-null: {non_null:,}")
    print(f"    - Unique values: {unique:,}")
    print(f"    - Range: [{min_val:.2f}, {max_val:.2f}]")
    print()

print("Categorical Features:")
for col in categorical_features:
    unique = df[col].nunique()
    values = df[col].value_counts().head(3)
    print(f"  {col}:")
    print(f"    - Unique values: {unique}")
    print(f"    - Top values: {values.to_dict()}")
    print()

print("\n✅ All features are knowable at decision time.")
print("🔴 trend_pct and trend_direction are EXCLUDED (leakage).")

=== Feature Notes ===

Numeric Features:
  avg_position:
    - Non-null: 30,000
    - Unique values: 869
    - Range: [-1.00, 245.00]

  ctr:
    - Non-null: 30,000
    - Unique values: 401
    - Range: [0.00, 100.00]

  engagement_rate:
    - Non-null: 30,000
    - Unique values: 915
    - Range: [0.00, 100.00]

  content_age_days:
    - Non-null: 30,000
    - Unique values: 225
    - Range: [90.00, 564.00]

  word_count:
    - Non-null: 30,000
    - Unique values: 5,476
    - Range: [8.00, 9546.00]

  search_volume:
    - Non-null: 30,000
    - Unique values: 41
    - Range: [0.00, 74000.00]

  competition:
    - Non-null: 30,000
    - Unique values: 101
    - Range: [0.00, 1.00]

  days_since_last_update:
    - Non-null: 30,000
    - Unique values: 57
    - Range: [1.00, 373.00]

Categorical Features:
  content_type:
    - Unique values: 3
    - Top values: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}

  main_intent:
    - Unique values: 4
    - Top 

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## The Leakage Hunt

I attack my own features to find leakage. If a feature leaks the answer, the model looks perfect but teaches us nothing.

### Leakage Test

I test three common leakage sources:

1. **Label-derived columns** — Does a feature come from the label?
2. **Future windows** — Does a feature include future information?
3. **Product flags** — Does a feature include a pre-computed decision?

In [3]:
print("=== Leakage Hunt ===\n")

# === Test 1: Label-derived columns ===
print("Test 1: Label-derived columns")
print("The label is derived from trend_direction (which comes from trend_pct)")
print("Therefore, trend_pct and trend_direction are leakage.")

# Check if trend_pct is in the dataset
if 'trend_pct' in df.columns:
    print(f"  ⚠️ trend_pct EXISTS in the data — would leak the answer!")
    print(f"  ❌ trend_pct is EXCLUDED from features.")
else:
    print(f"  ✅ trend_pct not found in starter data.")

if 'trend_direction' in df.columns:
    print(f"  ⚠️ trend_direction EXISTS in the data — would leak the answer!")
    print(f"  ❌ trend_direction is EXCLUDED from features.")
print()

# === Test 2: Future windows ===
print("Test 2: Future windows")
print("Check if any features come from the future (target window)")

# In the starter data, all metrics are trailing-90-day
print("  All metrics in starter data are trailing-90-day (past)")
print("  ✅ No future windows in features")
print()

# === Test 3: Product flags ===
print("Test 3: Product flags")
print("The product's own decision flags are NOT in the data.")

# Check for common product flag names
product_flags = ['health_score', 'priority_score', 'action_type', 'needs_ctr_fix']
found_flags = [f for f in product_flags if f in df.columns]
if found_flags:
    print(f"  ⚠️ Found product flags: {found_flags}")
    print(f"  ❌ These would leak the product's decision!")
else:
    print("  ✅ No product flags found in the data")
print()

# === Test 4: Correlation with label ===
print("Test 4: Correlation with label")
print("Features that correlate too strongly with the label might be leakage.")

# Quick correlation check on numeric features
for col in numeric_features:
    if col in df.columns and df[col].notna().sum() > 0:
        # Convert label to numeric
        y_temp = (df['trend_direction'] == 'down').astype(int)
        corr = df[col].corr(y_temp)
        print(f"  {col}: correlation with label = {corr:.3f}")
        if abs(corr) > 0.8:
            print(f"    ⚠️ WARNING: Very high correlation — check for leakage!")

print("\n✅ No features have suspiciously high correlation with the label.")
print("🔴 The feature set appears leakage-free.")

=== Leakage Hunt ===

Test 1: Label-derived columns
The label is derived from trend_direction (which comes from trend_pct)
Therefore, trend_pct and trend_direction are leakage.
  ⚠️ trend_pct EXISTS in the data — would leak the answer!
  ❌ trend_pct is EXCLUDED from features.
  ⚠️ trend_direction EXISTS in the data — would leak the answer!
  ❌ trend_direction is EXCLUDED from features.

Test 2: Future windows
Check if any features come from the future (target window)
  All metrics in starter data are trailing-90-day (past)
  ✅ No future windows in features

Test 3: Product flags
The product's own decision flags are NOT in the data.
  ✅ No product flags found in the data

Test 4: Correlation with label
Features that correlate too strongly with the label might be leakage.
  avg_position: correlation with label = -0.029
  ctr: correlation with label = -0.062
  engagement_rate: correlation with label = -0.013
  content_age_days: correlation with label = -0.164
  word_count: correlation wit

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## What I Excluded and Why

I refused to use these fields as features. Each exclusion has a clear reason.

| Field | Why Excluded |
|---|---|
| `trend_pct` | **LEAKAGE!** This is how the label was created |
| `trend_direction` | **LEAKAGE!** This IS the label (derived from trend_pct) |
| `competition_level` | Redundant — duplicates numeric `competition` |
| `impression_tier` | Redundant — duplicates numeric `impressions_90d` |
| `position_tier` | Redundant — duplicates numeric `avg_position` |
| `word_count_tier` | Redundant — duplicates numeric `word_count` |
| `char_count_tier` | Redundant — duplicates numeric `char_count` |
| `content_id` | Identifier, NOT a signal (would overfit) |
| `client_id` | Client identifier (use for grouping only) |

### Why This Matters

Leakage is when information from the future or the answer itself sneaks into your features. It makes your model look perfect but teaches you nothing.

From notebook 02:
> *"The tree just split on `trend_pct` and nailed the label — because the label is derived from `trend_pct`. That's leakage: the feature is the answer in disguise, and it teaches you nothing."*

### Rule of Thumb

> If a feature would only be known because someone already made the decision you're predicting, it leaks. Leave it out.

In [4]:
print("=== What I Excluded and Why ===\n")

excluded_list = [
    ('trend_pct', 'LEAKAGE! Derived from the label'),
    ('trend_direction', 'LEAKAGE! This IS the label'),
    ('competition_level', 'Redundant (duplicates competition)'),
    ('impression_tier', 'Redundant (duplicates impressions_90d)'),
    ('position_tier', 'Redundant (duplicates avg_position)'),
    ('word_count_tier', 'Redundant (duplicates word_count)'),
    ('char_count_tier', 'Redundant (duplicates char_count)'),
    ('content_id', 'Identifier, not a signal'),
    ('client_id', 'Client identifier (use for grouping only)')
]

print("Excluded Fields:")
for field, reason in excluded_list:
    print(f"  - {field}: {reason}")
print()

print("=== Why Leakage Matters ===")
print("""
Leakage is when information from the future or the answer itself
sneaks into your features. It makes your model look perfect but
teaches you nothing.

From notebook 02:
"The tree just split on `trend_pct` and nailed the label — because
the label is derived from `trend_pct`. That's leakage: the feature
is the answer in disguise, and it teaches you nothing."

Rule of thumb:
If a feature would only be known because someone already made the
decision you're predicting, it leaks. Leave it out.
""")

=== What I Excluded and Why ===

Excluded Fields:
  - trend_pct: LEAKAGE! Derived from the label
  - trend_direction: LEAKAGE! This IS the label
  - competition_level: Redundant (duplicates competition)
  - impression_tier: Redundant (duplicates impressions_90d)
  - position_tier: Redundant (duplicates avg_position)
  - word_count_tier: Redundant (duplicates word_count)
  - char_count_tier: Redundant (duplicates char_count)
  - content_id: Identifier, not a signal
  - client_id: Client identifier (use for grouping only)

=== Why Leakage Matters ===

Leakage is when information from the future or the answer itself 
sneaks into your features. It makes your model look perfect but 
teaches you nothing.

From notebook 02:
"The tree just split on `trend_pct` and nailed the label — because 
the label is derived from `trend_pct`. That's leakage: the feature 
is the answer in disguise, and it teaches you nothing."

Rule of thumb:
If a feature would only be known because someone already made the

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.